# End-to-End Cross-modal Multimodal RAG  with Google Gemini, LangChain & FAISS

This notebook builds a complete Multimodal Retrieval Augmented Generation (RAG) system.

The core idea is simple. A picture of a product is shown to Google Gemini. Gemini looks at
the picture and writes a detailed text description of what it sees. That description becomes
the knowledge base for a normal text based RAG pipeline. The description is split into small
chunks, each chunk is converted into a vector using an embedding model, and all the vectors
are stored inside a FAISS vector database. When a user asks a question, either by typing text
or by uploading a new image, the system searches FAISS for the most relevant chunks and asks
Gemini to answer the question using only that retrieved information.

In short, the notebook follows this linear flow:

1. Install the required Python packages
2. Import the required libraries
3. Configure the Google Gemini API key
4. Create a helper function that loads a Gemini chat model through LangChain
5. Load the Gemini text model and run a quick sanity check
6. Download and display a product image
7. Create the Google GenAI client that will be used for image understanding
8. Ask Gemini for a short summary of the image
9. Ask Gemini for a full, structured description of the image and save it as a text file
10. Load that text file with LangChain's TextLoader
11. Split the loaded text into small overlapping chunks
12. Convert the chunks into embeddings and store them inside a FAISS vector database
13. Build a retriever on top of FAISS and test it with a plain text query
14. Build the full Retrieval Augmented Generation chain (retriever plus prompt plus Gemini)
15. Ask the RAG chain a text question and see the grounded answer
16. Combine everything into one Multimodal RAG function that accepts an image directly
17. Test the full Multimodal RAG pipeline on a brand new product image

Every section below has a short explanation in a markdown cell followed by the code that
implements it, and the code itself is commented so the purpose of each line is clear even to
someone who has never seen this pipeline before.

## Step 1: Install the required libraries

Before any code can run we need a handful of Python packages.

LangChain is the framework that lets us connect language models, prompts, retrievers and
vector stores together instead of wiring all of that logic by hand.

langchain google genai is the connector package that allows LangChain to talk to Google's
Gemini models.

google genai is Google's own official SDK. It is used later for the image understanding part
of the pipeline because it has the most up to date support for sending images to Gemini.

faiss cpu is a fast similarity search library from Meta AI Research. It is used here to store
and search the text embeddings that represent our knowledge base.

langchain huggingface gives us access to open source embedding models that run locally, so we
do not need a paid embedding API.

python dotenv lets us load secret values such as API keys from a local .env file instead of
typing them directly into the notebook.

Run the cell below once per environment. In Google Colab this installs the packages into the
current session.

In [ ]:
%pip install --upgrade langchain langchain-google-genai google-genai langchain-community \
    langchain-huggingface langchain-text-splitters faiss-cpu python-dotenv pypdf


## Step 2: Import the libraries we will use

The imports are grouped by purpose so it is easy to see what each group is for.

The first group (os, requests, PIL) handles environment variables and downloading and opening
image files.

The second group (matplotlib, IPython.display) is used purely to show images and nicely
formatted markdown output inside the notebook. Neither library is part of the actual RAG
logic.

The third group brings in everything needed to talk to Gemini, split text, create embeddings
and run vector search with FAISS.

In [ ]:
# Standard library and basic utilities
import os
import requests
from dotenv import load_dotenv

# Used to open, save and display images
from PIL import Image
import matplotlib.pyplot as plt

# Used to show nicely formatted text (markdown) inside the notebook
from IPython.display import display, Markdown


In [ ]:
# LangChain wrapper around Gemini chat models (used for text generation and for RAG)
from langchain_google_genai import ChatGoogleGenerativeAI

# Google's official SDK, used later for sending an image plus a prompt to Gemini
from google import genai

# LangChain building blocks for prompts, chains and parsing model output
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LangChain building blocks for loading, chunking and representing text documents
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# The embedding model and the FAISS vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


## Step 3: Configure the Google Gemini API key

Gemini models are accessed through an API key generated in Google AI Studio.

Instead of typing the key directly into the notebook, it is kept inside a local file named
.env in the same folder as this notebook, on a line that looks like this:

GOOGLE_API_KEY=your_key_here

The load_dotenv function reads that file and loads its values as environment variables for
the current Python process. Both LangChain's ChatGoogleGenerativeAI and Google's own genai
SDK automatically look for a variable named GOOGLE_API_KEY, so once it is loaded here every
later cell in the notebook can use Gemini without repeating the key.

In [ ]:
# Load variables defined inside a local .env file into the environment
load_dotenv()

# Read the key so we can confirm it was actually found
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Make sure the key is present in the environment under the exact name that
# the Google libraries expect
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Google API key loaded:", GOOGLE_API_KEY is not None)


## Step 4: A small helper function that loads a Gemini chat model

Rather than repeating the same setup code every time we need a Gemini model, this helper
function takes a model name as a string and returns a ready to use chat model object.

Setting temperature to 0 tells the model to behave as deterministically as possible, which is
useful for a RAG system because we generally want consistent, grounded answers rather than
creative or random ones.

Note that Google occasionally renames or upgrades the available Gemini models. If a cell in
this notebook fails with a message saying the model could not be found, simply replace the
model name below with whichever current Gemini model name is listed in Google AI Studio.

In [ ]:
def load_model(model_name):
    """Return a LangChain chat model object connected to the given Gemini model."""
    return ChatGoogleGenerativeAI(
        model=model_name,
        temperature=0
    )


## Step 5: Load and test the text model

First we load a plain text Gemini model using the helper function above, then send it a
simple prompt just to confirm that the connection to Gemini is working end to end before we
build anything more complicated on top of it.

In [ ]:
# Load a Gemini text model through LangChain
model_text = load_model("gemini-3.5-flash")

# Send a simple prompt and print the reply, purely as a connectivity check
response = model_text.invoke("Tell me a short joke about software engineers.")
print(response.content)


## Step 6: Download and display a product image

Before Gemini can describe a product, we need an actual image to give it. The helper function
below downloads an image from a URL, saves a local copy inside a data/images folder, opens the
saved file with Pillow, and returns the resulting image object so it can be displayed in the
notebook and later handed to Gemini.

The function only does four things in order: it downloads the image, saves it to disk, opens
it with Pillow, and returns the Pillow Image object.

In [ ]:
def get_image(url, filename, extension):
    """
    Download an image from the given url and save it locally under
    <current_project>/data/images/<filename>.<extension>

    Returns a PIL.Image.Image object so it can be displayed or passed to Gemini.
    """

    # Folder where downloaded images will be stored
    project_dir = os.getcwd()
    image_dir = os.path.join(project_dir, "data", "images")
    os.makedirs(image_dir, exist_ok=True)

    # Full path of the file we are about to create
    image_path = os.path.join(image_dir, f"{filename}.{extension}")

    # Download the raw image bytes from the internet
    response = requests.get(url)
    response.raise_for_status()

    # Write the downloaded bytes to disk
    with open(image_path, "wb") as f:
        f.write(response.content)

    # Open the saved file so we get back a usable image object
    image = Image.open(image_path)

    print(f"Image saved at: {image_path}")
    return image


In [ ]:
# Download a Nike Dunk Low product photo to use as our first example
image = get_image(
    "https://static.nike.com/a/images/t_PDP_1728_v1/f_auto,q_auto:eco/1705ca64-fbc8-4b79-a451-4ab77760c219/dunk-low-older-shoes-C7T1cx.png",
    "nike_shoes",
    "png"
)

# Show the image directly inside the notebook
plt.imshow(image)
plt.axis("off")
plt.show()


## Step 7: Create a Google GenAI client for image understanding

LangChain's ChatGoogleGenerativeAI works well for text only tasks such as chatbots, prompt
chains and RAG, and we already used it above. For sending an image together with a prompt to
Gemini, however, this notebook uses Google's own official genai SDK instead.

There are three practical reasons for that choice.

First, it is the official SDK maintained directly by Google and tends to support new Gemini
capabilities before other libraries catch up.

Second, it accepts a Pillow Image object directly, so there is no need to manually convert the
picture into base64 text.

Third, the call itself is a single simple function, generate_content, which keeps the vision
part of the notebook easy to follow.

LangChain is still used everywhere else in this notebook: for the text model, the prompt
templates, and the full retrieval chain. Only the image understanding step uses the genai
SDK directly.

In [ ]:
# Create a client object that talks directly to the Gemini API for multimodal requests
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)


## Step 8: Ask Gemini for a short summary of the image

As a quick check that image understanding is working, we send the downloaded product image to
Gemini together with a very short instruction and print whatever text comes back.

In [ ]:
# A short instruction asking Gemini to summarize the picture
summary_prompt = "Give me a summary of this image in five words."

# generate_content accepts a list that mixes plain text and a PIL image in any order
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=[summary_prompt, image]
)

print(response.text)


## Step 9: Generate a detailed text description of the image and save it to a file

A vector database stores and searches text, not raw pixels, so before anything can be stored
in FAISS the image has to be turned into text.

Here we ask Gemini to act as a product analyst and describe the image in a structured way,
covering things such as the product name, brand, category, colors, material, design, any
visible logo, background, and a general description. If a particular detail simply is not
visible in the picture, Gemini is instructed to say so rather than guess, which helps keep the
generated knowledge base accurate.

The resulting description is written to a plain text file named nike_shoes.txt inside a data
folder. That file becomes the source document for every retrieval step that follows.

In [ ]:
# A detailed prompt asking Gemini to describe the product in a structured way
description_prompt = """
You are an expert product analyst.

Analyze the given image and generate a comprehensive description.

Include the following information:

Product Name
Brand
Category
Primary Colors
Secondary Colors
Material
Design
Visible Logo
Background
Objects Present
Possible Use
Detailed Description

If any information is not visible in the image, write:
"Not visible in the image."

Do not guess or invent missing information.
"""

# Send the prompt and the image to Gemini together
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=[description_prompt, image]
)

# The generated description is plain text
image_description = response.text

# Make sure the output folder exists
output_dir = os.path.join(os.getcwd(), "data")
os.makedirs(output_dir, exist_ok=True)

# Save the description so it can be loaded again like any normal text document
text_file = os.path.join(output_dir, "nike_shoes.txt")
with open(text_file, "w", encoding="utf-8") as file:
    file.write(image_description)

print(f"Image description saved to: {text_file}")
print()
print(image_description)


The overall idea at this point can be pictured as a short pipeline:

Image goes into Gemini (vision model), Gemini produces a detailed text description, that
description is saved as nike_shoes.txt, and everything from here on works with that text
file exactly like a normal text based RAG system would.

## Step 10: Load the saved description with LangChain's TextLoader

The retrieval part of a Multimodal RAG system searches over text, not images. Now that the
image has been converted into a text file in the previous step, we load that file using
LangChain's TextLoader, which reads the file from disk and wraps its content inside a
LangChain Document object. That Document is what the rest of the pipeline (splitting,
embedding, storing) expects to work with.

In [ ]:
# Path to the description file we created in the previous step
text_path = os.path.join(os.getcwd(), "data", "nike_shoes.txt")

# TextLoader reads the file and returns a list of Document objects
loader = TextLoader(text_path)
documents = loader.load()

# Show the text that was loaded, to confirm it matches what we saved
print(documents[0].page_content)


## Step 11: Split the text into small overlapping chunks

Embedding models and retrievers generally work better on short, focused pieces of text rather
than one long document. This step uses LangChain's CharacterTextSplitter to break the product
description into smaller chunks, each around 300 characters long, with a 50 character overlap
between neighboring chunks.

The overlap matters because it helps make sure that a sentence which happens to fall right on
a chunk boundary is not cut in half in a way that loses its meaning. Each resulting chunk is
wrapped in a LangChain Document object so it can be embedded and stored in the next step.

In [ ]:
def get_text_chunks_langchain(text):
    """Split a long piece of text into small overlapping chunks and wrap each
    chunk inside a LangChain Document object."""

    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=300,
        chunk_overlap=50,
        length_function=len,
    )

    chunks = text_splitter.split_text(text)
    docs = [Document(page_content=chunk) for chunk in chunks]
    return docs


In [ ]:
# Split the loaded description into chunks
docs = get_text_chunks_langchain(documents[0].page_content)

print(f"Number of chunks created: {len(docs)}")
print()
print("First chunk:")
print(docs[0].page_content)


## Step 12: Generate embeddings and store them inside FAISS

An embedding model converts a piece of text into a dense numerical vector that captures its
meaning, so that pieces of text with similar meaning end up with vectors that are close to
each other in that vector space.

Instead of calling a paid embedding API, this notebook uses the open source BAAI/bge base en
v1.5 embedding model from Hugging Face, which runs locally. Using a local embedding model has
a few practical advantages: it is free to run, it has no request quota, and it produces high
quality general purpose embeddings that work well for RAG.

Once every chunk has been embedded, the vectors are stored inside a FAISS vector database.
FAISS is a fast similarity search library that, given a query vector, can quickly find the
stored vectors that are closest to it, which is exactly what a retriever needs to do.

In [ ]:
# Load the embedding model. The first run downloads the model weights locally.
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)


In [ ]:
# Build a FAISS index directly from the chunked documents. This embeds every
# chunk and stores the resulting vectors together with the original text.
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

print("FAISS vector store created successfully with", len(docs), "chunks.")


## Step 13: Create a retriever and test it

A retriever is a thin wrapper around the vector store. Given a plain text query, it embeds
that query using the same embedding model used above, searches FAISS for the closest stored
vectors, and returns the matching chunks of text.

Here we create a retriever that returns the top 3 most similar chunks for any given query, and
test it with two simple questions to confirm it returns sensible, relevant text.

In [ ]:
# Wrap the FAISS vector store in a retriever interface
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Try a first test query
results = retriever.invoke("Nike slide or sandal")
for i, doc in enumerate(results, 1):
    print(f"Result {i}")
    print(doc.page_content)
    print()


In [ ]:
# Try a second test query
query = "Which brand is this product?"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"Result {i}")
    print(doc.page_content)
    print()


## Step 14: Build the Retrieval Augmented Generation chain

This is the step where retrieval and generation are combined into a single pipeline, usually
called a chain.

When a user submits a question, the chain does the following, in order:

1. The question is passed to the retriever, which embeds it and searches FAISS for the most
   relevant chunks of text.
2. Those retrieved chunks, along with the original question, are inserted into a prompt
   template.
3. The completed prompt is sent to the Gemini text model.
4. Gemini writes an answer based only on the retrieved context, and is instructed to say so
   plainly if the answer is not contained in that context.
5. The raw model output is parsed into a plain text string for easy display.

Grounding the answer in retrieved text like this is what reduces hallucination, since Gemini
is being asked to answer using the specific information we retrieved rather than relying
purely on what it may already know from training.

In [ ]:
# Load a Gemini text model to use for the generation step of the RAG chain
llm_text = load_model("gemini-3.5-flash")


In [ ]:
# The prompt template defines exactly how the retrieved context and the
# user's question are combined before being sent to Gemini
template = """
You are a helpful AI assistant.

Use ONLY the retrieved context below to answer the user's question.

If the answer is not present in the context, reply:
"I couldn't find that information in the knowledge base."

Retrieved Context:
{context}

User Question:
{query}

Provide a clear, concise and accurate answer.
"""

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
# The full RAG chain, expressed as a pipeline of steps joined with the pipe operator
#
# 1. "context" is filled in by calling the retriever with the incoming query
# 2. "query" is passed through unchanged with RunnablePassthrough
# 3. the resulting dictionary is formatted into the prompt template
# 4. the completed prompt is sent to the Gemini text model
# 5. StrOutputParser converts the model's response object into a plain string
rag_chain = (
    {
        "context": retriever,
        "query": RunnablePassthrough()
    }
    | prompt
    | llm_text
    | StrOutputParser()
)


## Step 15: Ask the RAG pipeline a text question

The pipeline is now ready to answer questions grounded in the product description we stored
earlier. We send it a plain text question and display the answer as formatted markdown.

In [ ]:
query = "Describe the Nike shoe in detail."

result = rag_chain.invoke(query)

display(Markdown(result))


## Step 16: Combine everything into one Multimodal RAG function

Up to this point the pipeline answered a typed text question. The final piece is to let a user
provide an image instead of typing a question at all.

The complete Multimodal RAG pipeline works in two stages:

1. Image understanding: the input image is sent to Gemini, which returns a detailed text
   description of it, exactly like the step used earlier to build the knowledge base.
2. Retrieval and generation: that generated description is then used as the query for the RAG
   chain built above, so the retriever searches FAISS for relevant chunks and Gemini writes
   the final answer using that retrieved context.

The function below wraps both stages together so that a single call, given only an image,
returns a final, context aware answer.

Visually the flow looks like this:

Input image goes into Gemini (vision), Gemini produces a text description, that description
becomes the query, the RAG retriever searches FAISS with it, the retrieved chunks and the
description are combined inside the prompt template, Gemini (text) generates the final
answer.

In [ ]:
def describe_image_with_gemini(pil_image, prompt_text=None):
    """Send an image to Gemini and return a detailed text description of it."""

    if prompt_text is None:
        prompt_text = """
Analyze this product image in detail.

Include:
Product Name
Brand
Category
Colors
Material
Design
Visible Logo
Intended Use

Write a detailed description.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=[prompt_text, pil_image]
    )

    return response.text


def multimodal_rag(pil_image):
    """Full Multimodal RAG pipeline.

    Step one, describe the given image using Gemini's vision capability.
    Step two, feed that description into the existing text based RAG chain so the
    retriever can search FAISS and Gemini can generate a grounded final answer.
    """

    # Stage 1: turn the image into a detailed text description
    image_description = describe_image_with_gemini(pil_image)

    # Stage 2: use that description as the query for the RAG chain
    final_answer = rag_chain.invoke(image_description)

    return image_description, final_answer


## Step 17: Test the complete Multimodal RAG pipeline on a new image

Finally we download a second product image, a pair of Nike slides, and run it through the
multimodal_rag function defined above. No text question is typed by the user at all. Gemini
first looks at the image and writes a description of it, and that description alone drives
the retrieval and the final generated answer.

In [ ]:
# Download a second product image, a pair of Nike slides
slide_url = "https://static.nike.com/a/images/t_PDP_1728_v1/f_auto,q_auto:eco/252f2db6-d426-4931-80a0-8b7f8f875536/calm-slides-K7mr3W.png"

image_2 = get_image(slide_url, "nike_slides", "png")

plt.imshow(image_2)
plt.axis("off")
plt.show()


In [ ]:
# Run the full Multimodal RAG pipeline on the new image
image_description, final_answer = multimodal_rag(image_2)

print("Generated image description:")
print(image_description)
print()
print("Final answer from the RAG pipeline:")
display(Markdown(final_answer))


## Conclusion

This notebook built a complete Multimodal Retrieval Augmented Generation pipeline that
combines image understanding, semantic search and a large language model.

The workflow converts an image into searchable text using Google Gemini, splits and embeds
that text, stores the resulting vectors inside a FAISS vector database, retrieves the most
relevant chunks for a given query, and finally asks Gemini to generate an answer grounded in
that retrieved context. Because the final answer is required to come from the retrieved text
rather than only from what the model already knows, this approach reduces hallucination and
keeps the answers tied to the actual product information that was captured from the image.

Key takeaways:

1. Images are converted into detailed textual descriptions using Gemini's vision capability.
2. The generated text becomes the searchable knowledge base for the rest of the pipeline.
3. The text is split into small overlapping chunks before being embedded.
4. A local, open source embedding model (BAAI/bge base en v1.5) turns each chunk into a
   vector, and FAISS stores those vectors for fast similarity search.
5. A retriever finds the chunks most relevant to a given question or image description.
6. Gemini generates the final answer using only the retrieved context, which is combined with
   the question inside a prompt template.
7. Wrapping the vision step and the RAG chain together produces a single function that can
   answer questions directly from a new image, with no typed question required.